# Car Racing (CarRacing-v3) 环境介绍

**Car Racing** 是 Gymnasium 库中一个经典的 Box2D 环境，任务是通过像素输入控制赛车，通过俯视视角进行驾驶。

这是一个从像素学习控制策略（Learning from Pixels）的直观任务，每一轮生成的赛道都是随机的。

In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
from IPython.display import Video

# 创建环境，指定渲染模式
env = gym.make("CarRacing-v3", render_mode="rgb_array")

print(f"环境名称: {env.spec.id}")
print(f"动作空间类型: {env.action_space}")

环境名称: CarRacing-v3
动作空间类型: Box([-1.  0.  0.], 1.0, (3,), float32)


c:\Users\pc\AppData\Local\Programs\Python\Python39\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## 1. 观察空间 (Observation Space)

Agent 接收到的输入是一张 **96x96 的 RGB 图像**。

状态空间是一个形状为 `(96, 96, 3)` 的数组，数值范围在 0 到 255 之间。

图像底部包含一些仪表盘信息，从左到右依次为：真实速度、四个ABS传感器、方向盘位置和陀螺仪。

In [2]:
# 重置环境以获取初始帧
observation, info = env.reset()

print(f"Observation Shape: {observation.shape}")
print(f"Observation Data Type: {observation.dtype}")
print(f"Observation Value Range: [{observation.min()}, {observation.max()}]")

Observation Shape: (96, 96, 3)
Observation Data Type: uint8
Observation Value Range: [0, 228]


<img src="./car.png" width=900>

## 2. 动作空间 (Action Space)

CarRacing-v3 使用连续动作空间，由 3 个连续的分量组成：

1. **Steering (方向盘)**: 范围 `[-1, 1]`。`-1` 是最左，`+1` 是最右。
2. **Gas (油门)**: 范围 `[0, 1]`。
3. **Brake (刹车)**: 范围 `[0, 1]`。

> **注意**: 这是一款大马力的后驱车，请避免同时踩油门和转向，否则容易失控。

In [3]:
print(f"Action Space: {env.action_space}")
print(f"Action Space Shape: {env.action_space.shape}")
print(f"Action Low: {env.action_space.low}")
print(f"Action High: {env.action_space.high}")

# 测试一个示例动作
example_action = [0.0, 0.5, 0.0]  # 直行，中等油门，不刹车
print(f"示例动作: {example_action}")

Action Space: Box([-1.  0.  0.], 1.0, (3,), float32)
Action Space Shape: (3,)
Action Low: [-1.  0.  0.]
Action High: [1. 1. 1.]
示例动作: [0.0, 0.5, 0.0]


如果是离散的，则有5个动作：

0: do nothing  0：什么都不做

1: steer right  1：向右转

2: steer left  2：向左转

3: gas  3：燃气

4: brake  4：刹车

## 3. 奖励机制 (Rewards)

目标是在尽可能短的时间内跑完赛道。奖励由以下部分组成：

* **访问奖励**: 每访问一个新的赛道地块（Tile），获得 `+1000/N` 的奖励（N 是总地块数）。跑完一圈的总奖励为 1000 分。
* **时间惩罚**: 每一帧减少 `-0.1` 的奖励。
* **出界惩罚**: 如果车辆开出游戏区域，奖励 `-100` 并结束游戏。

**例子**: 如果你在 732 帧内跑完全程，最终得分计算为：$1000 - 0.1 \times 732 = 926.8$ 分。

## 4. 环境交互演示

In [6]:
import cv2
import imageio

def add_text_to_frame(frame, step, total_reward):
    """
    在帧上添加步数和奖励信息
    
    Args:
        frame: 原始帧
        step: 当前步数
        total_reward: 总奖励
    
    Returns:
        添加文本后的帧
    """
    # 复制帧以避免修改原始数据
    frame_with_text = frame.copy()
    
    # 添加文本信息
    text_step = f"Step: {step+1}"
    text_reward = f"Reward: {total_reward:.2f}"
    
    # 设置文本位置
    position_step = (10, 30)
    position_reward = (10, 60)
    
    # 设置文本属性
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    color = (255, 255, 255)  # 白色
    thickness = 2
    
    # 添加文本到帧
    cv2.putText(frame_with_text, text_step, position_step, font, font_scale, color, thickness)
    cv2.putText(frame_with_text, text_reward, position_reward, font, font_scale, color, thickness)
    
    return frame_with_text

def create_interaction_gif(env, steps=100, gif_name="car_racing_interaction.gif"):
    """
    创建环境交互的GIF动画
    
    Args:
        env: 环境实例
        steps: 交互步数
        gif_name: 保存的GIF文件名
    """
    # 创建用于渲染的环境
    render_env = gym.make('CarRacing-v3', continuous=True, render_mode='rgb_array')
    
    # 重置环境
    observation, info = render_env.reset()
    frames = []
    
    total_reward = 0
    
    for step in range(steps):
        # 随机动作
        action = render_env.action_space.sample()
        
        # 执行动作
        observation, reward, terminated, truncated, info = render_env.step(action)
        total_reward += reward
        
        # 获取当前帧并添加到帧列表
        frame = render_env.render()
        frames.append(frame)
        
        # 添加步数和奖励信息到帧上
        frame_with_text = add_text_to_frame(frame, step, total_reward)
        frames[-1] = frame_with_text
        
        if terminated or truncated:
            print(f"Episode finished after {step + 1} steps")
            break
    
    # 保存为GIF
    imageio.mimsave(gif_name, frames, fps=30)
    print(f"GIF saved as {gif_name}")
    print(f"总奖励: {total_reward:.2f}")
    
    render_env.close()
    
    return frames, total_reward

gif_frames, total_reward = create_interaction_gif(
    env, 
    steps=200,  # 增加步数以获得更长的GIF
    gif_name="car_racing_demo.gif"
)

GIF saved as car_racing_demo.gif
总奖励: -1.82


## 5. 赛车演示视频

以下是模型在环境中的实际运行表现：

In [5]:
# 请确保您的视频文件名为 'car_racing_demo.mp4' 并与此 notebook 在同一目录下
try:
    Video("./video.webm", width=600, height=400, embed=True)
except:
    print("未找到视频文件 'car_racing_demo.mp4'，请确保文件存在并与此 notebook 在同一目录下")

pip install swig

pip install gymnasium[box2d]